# Chapter 12 — SOLUTIONS: Capstone End-to-End ML Workflow

**Instructor file — share only after exercise session.**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

sns.set_theme(style='whitegrid')
np.random.seed(42)

print('All imports ready!')

## Step 1 — Load and Explore the Data

In [ ]:
df_raw = sns.load_dataset('titanic')
print('Shape:', df_raw.shape)
df_raw.head()

In [ ]:
# Solution 1a: dtypes and missing values
print('=== Data Types ===')
print(df_raw.dtypes)
print('\n=== Missing Values ===')
print(df_raw.isnull().sum())
print('\n=== Basic Stats ===')
print(df_raw.describe())

In [ ]:
# Solution 1b-d: EDA visualizations
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Survival by gender
sns.barplot(data=df_raw, x='sex', y='survived', ax=axes[0],
            palette={'male': '#3498db', 'female': '#e74c3c'})
axes[0].set_title('Survival Rate by Gender', fontsize=13)
axes[0].set_ylabel('Survival Rate')
axes[0].set_ylim(0, 1)
for p in axes[0].patches:
    axes[0].annotate(f'{p.get_height():.1%}',
                     (p.get_x() + p.get_width()/2., p.get_height()),
                     ha='center', va='bottom', fontsize=12)

# Survival by class
sns.barplot(data=df_raw, x='pclass', y='survived', ax=axes[1],
            palette='Blues_r')
axes[1].set_title('Survival Rate by Class', fontsize=13)
axes[1].set_ylabel('Survival Rate')
axes[1].set_ylim(0, 1)
for p in axes[1].patches:
    axes[1].annotate(f'{p.get_height():.1%}',
                     (p.get_x() + p.get_width()/2., p.get_height()),
                     ha='center', va='bottom', fontsize=12)

# Age distribution by survival
df_raw[df_raw['survived'] == 0]['age'].dropna().hist(
    ax=axes[2], bins=25, alpha=0.6, color='#e74c3c', label='Did not survive')
df_raw[df_raw['survived'] == 1]['age'].dropna().hist(
    ax=axes[2], bins=25, alpha=0.6, color='#2ecc71', label='Survived')
axes[2].set_title('Age Distribution by Survival', fontsize=13)
axes[2].set_xlabel('Age')
axes[2].set_ylabel('Count')
axes[2].legend()

plt.suptitle('Titanic — Exploratory Data Analysis', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print('Key observations:')
print(f'  Female survival rate: {df_raw[df_raw["sex"]=="female"]["survived"].mean():.1%}')
print(f'  Male survival rate:   {df_raw[df_raw["sex"]=="male"]["survived"].mean():.1%}')
for cls in [1, 2, 3]:
    rate = df_raw[df_raw['pclass']==cls]['survived'].mean()
    print(f'  Class {cls} survival rate: {rate:.1%}')

## Step 2 — Clean and Preprocess

In [ ]:
df = df_raw.copy()

# Solution 2a: Impute Age with median per class
df['age'] = df['age'].fillna(
    df.groupby('pclass')['age'].transform('median')
)
print('Missing age values after imputation:', df['age'].isnull().sum())

# Show the per-class medians used
print('Median age per class:')
print(df_raw.groupby('pclass')['age'].median())

In [ ]:
# Solution 2b: Fill missing Embarked with mode
most_common_port = df['embarked'].mode()[0]
df['embarked'] = df['embarked'].fillna(most_common_port)
print(f'Filled missing embarked with: "{most_common_port}"')
print('Missing embarked values:', df['embarked'].isnull().sum())

In [ ]:
# Solution 2c: Drop redundant / derived / leaky columns
cols_to_drop = ['deck', 'embark_town', 'alive', 'who', 'adult_male', 'alone']
df.drop(columns=cols_to_drop, inplace=True)
print('Remaining columns:', df.columns.tolist())
print('Any remaining missing values?')
print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
# Solution 2d: Encode categorical variables

# Binary encode sex
df['sex'] = (df['sex'] == 'male').astype(int)
print('Sex encoded: male=1, female=0')

# One-hot encode embarked
df = pd.get_dummies(df, columns=['embarked'], drop_first=True)

print('Shape after encoding:', df.shape)
print('Columns:', df.columns.tolist())
df.head(3)

## Step 3 — Feature Engineering

In [ ]:
# Solution 3a: FamilySize
df['family_size'] = df['sibsp'] + df['parch'] + 1
print('FamilySize distribution:')
print(df['family_size'].value_counts().sort_index())

In [ ]:
# Solution 3b: IsAlone
df['is_alone'] = (df['family_size'] == 1).astype(int)
print(f"Fraction traveling alone: {df['is_alone'].mean():.1%}")

In [ ]:
# Solution 3c: Survival by FamilySize
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Survival rate by family size
family_survival = df.groupby('family_size')['survived'].mean()
axes[0].bar(family_survival.index, family_survival.values, color='#3498db', alpha=0.8)
axes[0].set_xlabel('Family Size')
axes[0].set_ylabel('Survival Rate')
axes[0].set_title('Survival Rate by Family Size', fontsize=13)
axes[0].axhline(df['survived'].mean(), color='red', linestyle='--',
                label=f'Overall: {df["survived"].mean():.1%}')
axes[0].legend()

# Alone vs not alone
alone_labels = ['Not Alone', 'Alone']
alone_survival = df.groupby('is_alone')['survived'].mean()
axes[1].bar(alone_labels, alone_survival.values,
            color=['#2ecc71', '#e74c3c'], alpha=0.8)
axes[1].set_ylabel('Survival Rate')
axes[1].set_title('Survival Rate: Alone vs Not Alone', fontsize=13)
for i, v in enumerate(alone_survival.values):
    axes[1].text(i, v + 0.01, f'{v:.1%}', ha='center', fontsize=12)

plt.tight_layout()
plt.show()

print('\nInsight: Small families (2-4) had higher survival rates.')
print('Very large families and solo travelers had lower rates.')

## Step 4 — Train and Evaluate Multiple Models

In [ ]:
# Solution 4a: Prepare X and y
X = df.drop(columns=['survived'])
y = df['survived']

print('X shape:', X.shape)
print('y shape:', y.shape)
print('Survival rate:', y.mean().round(3))
print('Features:', X.columns.tolist())

In [ ]:
# Solution 4b: Cross-validate three models using Pipelines (no leakage)
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=100, random_state=42),
}

results = {}
for name, model in models.items():
    # Wrap in Pipeline: scaler is fit inside each CV fold → no leakage
    pipe = Pipeline([('scaler', StandardScaler()), ('clf', model)])
    f1_scores  = cross_val_score(pipe, X, y, cv=5, scoring='f1')
    acc_scores = cross_val_score(pipe, X, y, cv=5, scoring='accuracy')
    results[name] = {
        'F1 Mean':  f1_scores.mean(),
        'F1 Std':   f1_scores.std(),
        'Acc Mean': acc_scores.mean(),
        'Acc Std':  acc_scores.std(),
    }
    print(f'{name}:')
    print(f'  F1:       {f1_scores.mean():.3f} ± {f1_scores.std():.3f}')
    print(f'  Accuracy: {acc_scores.mean():.3f} ± {acc_scores.std():.3f}')
    print()

In [ ]:
# Solution 4c: Bar chart comparison
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

model_names = list(results.keys())
f1_means  = [results[m]['F1 Mean']  for m in model_names]
f1_stds   = [results[m]['F1 Std']   for m in model_names]
acc_means = [results[m]['Acc Mean'] for m in model_names]
acc_stds  = [results[m]['Acc Std']  for m in model_names]

colors = ['#3498db', '#2ecc71', '#e74c3c']

# F1
bars = axes[0].bar(model_names, f1_means, yerr=f1_stds,
                   color=colors, alpha=0.8, capsize=5)
axes[0].set_title('F1-Score (5-fold CV)', fontsize=13)
axes[0].set_ylabel('F1-Score')
axes[0].set_ylim(0, 1)
axes[0].tick_params(axis='x', rotation=15)
for bar, val in zip(bars, f1_means):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f'{val:.3f}', ha='center', fontsize=11)

# Accuracy
bars = axes[1].bar(model_names, acc_means, yerr=acc_stds,
                   color=colors, alpha=0.8, capsize=5)
axes[1].set_title('Accuracy (5-fold CV)', fontsize=13)
axes[1].set_ylabel('Accuracy')
axes[1].set_ylim(0, 1)
axes[1].tick_params(axis='x', rotation=15)
for bar, val in zip(bars, acc_means):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f'{val:.3f}', ha='center', fontsize=11)

plt.suptitle('Model Comparison — Titanic Survival Prediction', fontsize=14)
plt.tight_layout()
plt.show()

## Step 5 — Interpret Results

In [ ]:
# Solution 5a: Feature importances from Random Forest
# Random Forest does not need scaling — it's tree-based
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X, y)

importances = pd.Series(rf.feature_importances_, index=X.columns)
importances = importances.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 6))
colors_imp = ['#e74c3c' if imp > importances.median() else '#95a5a6'
              for imp in importances.values]
ax.barh(importances.index, importances.values, color=colors_imp, alpha=0.85)
ax.set_xlabel('Feature Importance', fontsize=12)
ax.set_title('Random Forest — Feature Importances\n(Titanic Survival Prediction)', fontsize=13)
ax.axvline(importances.median(), color='navy', linestyle='--', alpha=0.5,
           label='Median importance')
ax.legend()
plt.tight_layout()
plt.show()

print('Top 3 most important features:')
for feat, imp in importances.sort_values(ascending=False).head(3).items():
    print(f'  {feat}: {imp:.3f}')

In [ ]:
# Solution 5b: Confusion matrix on held-out test set
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale using training data only (no leakage)
scaler_cv = StandardScaler()
X_train_sc = scaler_cv.fit_transform(X_train)
X_test_sc  = scaler_cv.transform(X_test)

# Train best model (typically Random Forest or Gradient Boosting)
best_model = GradientBoostingClassifier(n_estimators=100, random_state=42)
best_model.fit(X_train_sc, y_train)
y_pred = best_model.predict(X_test_sc)

# Plot confusion matrix
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=['Did not survive', 'Survived'])
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion Matrix — Gradient Boosting\n(Test set)', fontsize=13)
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'True Negatives  (correctly predicted died):     {tn}')
print(f'False Positives (predicted survived, actually died): {fp}')
print(f'False Negatives (predicted died, actually survived): {fn}  ← cost!')
print(f'True Positives  (correctly predicted survived): {tp}')

In [ ]:
# Solution 5c: Classification report
print('Classification Report:')
print(classification_report(y_test, y_pred,
                             target_names=['Did not survive', 'Survived']))

## Summary — What We Found

### Key Findings

| Feature | Role in survival |
|---------|------------------|
| `sex` | **Strongest predictor** — women survived at ~74%, men at ~19% |
| `pclass` | **Strong** — 1st class ~63%, 3rd class ~24% |
| `age` | Children had higher survival rates |
| `fare` | Correlates with class; higher fare → higher survival |
| `family_size` | Small families survived best; solo travelers and large families worse |

### Model Performance

All three models achieved ~80% accuracy and F1 ≈ 0.75–0.78.  
Gradient Boosting and Random Forest typically outperform Logistic Regression slightly.

### What This Workflow Demonstrates

- **EDA first**: the gender and class signals were obvious before modeling
- **Thoughtful imputation**: per-class median age is more accurate than a global median
- **Feature engineering**: FamilySize captured something the raw SibSp/Parch columns didn't
- **Cross-validation**: gives honest estimates; train/test split for final evaluation
- **Interpretation matters**: knowing *why* a model predicts what it does is as important as the accuracy

### The Data Science Workflow — Complete

```
① Define Problem ✓     ② Load & Explore ✓     ③ Clean & Preprocess ✓
④ Feature Engineering ✓  ⑤ Train & Evaluate ✓  ⑥ Interpret & Reflect ✓
```